# Conway's Game of Life

Conway's **Game of Life** (1970) is a two-dimensional cellular automaton invented by mathematician John Horton Conway. It is zero-player: given an initial configuration, the entire future evolution is deterministic. Despite its four simple rules, it supports emergent behaviors of extraordinary complexity — gliders, oscillators, guns, and even Turing-complete computation.

The Game of Life is a foundational example in the study of **complex systems**: how local, simple interactions can give rise to global, complicated structure.

## The Rules

The state space is an (infinite) two-dimensional grid. Each cell $(i, j)$ is either **alive** ($= 1$) or **dead** ($= 0$). Let

$$N(i,j) = \sum_{(di,\,dj) \in \{-1,0,1\}^2 \setminus \{(0,0)\}} c_{i+di,\,j+dj}$$

denote the number of live cells in the **Moore neighborhood** (the 8 surrounding cells). The update rules are:

| Condition | Outcome |
|-----------|--------|
| Live cell, $N < 2$ | Dies (underpopulation) |
| Live cell, $N \in \{2, 3\}$ | Survives |
| Live cell, $N > 3$ | Dies (overpopulation) |
| Dead cell, $N = 3$ | Born (reproduction) |

Compactly: **B3/S23** (born with 3 neighbors, survives with 2 or 3).

All cells update **simultaneously** — each new generation is computed from the current generation, not from partially updated states.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")

## Efficient Neighbor Counting

A naive implementation loops over every cell and its 8 neighbors — $O(N^2 \cdot 8)$ per step. A much cleaner approach sums 8 **shifted copies** of the grid. If we shift the grid by $(\Delta i, \Delta j)$, the value at position $(i, j)$ in the shifted grid equals the original value at $(i - \Delta i, j - \Delta j)$, i.e., the neighbor in direction $(\Delta i, \Delta j)$.

```
N(i,j) = sum of shifted grids over (di, dj) in {-1,0,1}^2 \ {(0,0)}
```

`np.roll` performs cyclic shifts along any axis, automatically implementing **toroidal (periodic) boundary conditions** — cells that leave one edge reappear at the opposite edge.

In [ ]:
def count_neighbors(grid):
    """Count live Moore neighbors for every cell simultaneously."""
    N = np.zeros_like(grid, dtype=np.int32)
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            if di == 0 and dj == 0:
                continue
            N += np.roll(np.roll(grid, di, axis=0), dj, axis=1)
    return N


def gol_step(grid):
    """Apply one generation of Conway's Game of Life rules."""
    N = count_neighbors(grid)
    birth   = (grid == 0) & (N == 3)            # dead cell with exactly 3 neighbors
    survive = (grid == 1) & ((N == 2) | (N == 3)) # live cell with 2 or 3 neighbors
    return (birth | survive).astype(np.uint8)


def gol_run(grid, n_steps):
    """Evolve a grid for n_steps generations. Returns list of frames."""
    frames = [grid.copy()]
    for _ in range(n_steps):
        grid = gol_step(grid)
        frames.append(grid.copy())
    return frames

## Classic Patterns

### The Blinker
The **blinker** is the simplest oscillator: a horizontal row of 3 cells that alternates between horizontal and vertical orientations with **period 2**.

### The Glider
The **glider** is the simplest spaceship: a 5-cell pattern with **period 4** that translates diagonally by $(1, 1)$ every 4 generations, effectively "flying" across the grid.

### The Gosper Glider Gun
The **Gosper glider gun** (1970) was the first discovered pattern that grows indefinitely. It periodically emits gliders, one every 30 generations, producing an infinite stream. Its discovery resolved Conway's original question about whether any finite pattern could produce unbounded growth.

In [ ]:
def make_blinker(rows=15, cols=15):
    """Horizontal blinker centered on a rows x cols grid."""
    g = np.zeros((rows, cols), dtype=np.uint8)
    r, c = rows // 2, cols // 2
    g[r, c - 1:c + 2] = 1
    return g


def make_glider(rows=20, cols=20):
    """Classic glider near the top-left of a rows x cols grid."""
    g = np.zeros((rows, cols), dtype=np.uint8)
    # Standard glider pattern (row offsets, col offsets from origin)
    for r, c in [(0, 1), (1, 2), (2, 0), (2, 1), (2, 2)]:
        g[r + 2, c + 2] = 1
    return g


def make_gosper_gun(rows=40, cols=80):
    """Gosper glider gun on a rows x cols grid."""
    g = np.zeros((rows, cols), dtype=np.uint8)
    # Standard 36-cell Gosper glider gun coordinates
    gun_cells = [
        (5, 1), (5, 2), (6, 1), (6, 2),           # left square
        (5, 11), (6, 11), (7, 11),                 # left part of gun body
        (4, 12), (8, 12),
        (3, 13), (3, 14), (9, 13), (9, 14),
        (6, 15),
        (4, 16), (8, 16),
        (5, 17), (6, 17), (7, 17),
        (6, 18),
        (3, 21), (4, 21), (5, 21),                 # right part of gun body
        (3, 22), (4, 22), (5, 22),
        (2, 23), (6, 23),
        (1, 25), (2, 25), (6, 25), (7, 25),
        (3, 35), (4, 35), (3, 36), (4, 36),        # right square
    ]
    for r, c in gun_cells:
        if 0 <= r < rows and 0 <= c < cols:
            g[r, c] = 1
    return g

## Blinker: Period-2 Oscillation

The blinker alternates between a $1\times 3$ horizontal bar and a $3\times 1$ vertical bar. Generations 0 and 2 are identical; generations 1 and 3 are identical.

In [ ]:
blinker_frames = gol_run(make_blinker(), n_steps=3)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
fig.suptitle("Blinker — Period-2 Oscillator", fontsize=13)
for i, ax in enumerate(axes):
    ax.imshow(blinker_frames[i], cmap='binary_r', interpolation='nearest')
    ax.set_title(f'Gen {i}', fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Glider: Diagonal Translation

The glider completes a full cycle in 4 generations, having moved 1 cell diagonally. Its speed is $c/4$ where $c$ is the maximum signal speed (1 cell/generation) on the grid.

In [ ]:
glider_frames = gol_run(make_glider(rows=20, cols=20), n_steps=8)
gens_to_show = [0, 2, 4, 6, 8]

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
fig.suptitle("Glider — Period-4 Spaceship", fontsize=13)
for ax, g in zip(axes, gens_to_show):
    ax.imshow(glider_frames[g], cmap='binary_r', interpolation='nearest')
    ax.set_title(f'Gen {g}', fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Gosper Glider Gun: Unbounded Growth

The gun emits one glider every 30 generations. By generation 120 we can see four gliders in flight, moving toward the lower-right of the grid.

In [ ]:
gun_frames = gol_run(make_gosper_gun(rows=40, cols=80), n_steps=120)
gens_to_show = [0, 40, 80, 120]

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
fig.suptitle("Gosper Glider Gun — Periodic Glider Emission", fontsize=14)
for ax, g in zip(axes.flat, gens_to_show):
    ax.imshow(gun_frames[g], cmap='binary_r', interpolation='nearest', aspect='equal')
    ax.set_title(f'Generation {g}', fontsize=12)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Boundary Conditions and Finite Grids

Our implementation uses **toroidal (periodic) boundaries**: cells that exit the right edge of the grid reappear at the left, and similarly for top/bottom. This is mathematically convenient but physically unrealistic — on a true infinite grid, the gun would emit gliders indefinitely without them ever returning.

On a finite toroidal grid, gliders will eventually wrap around and collide with the gun or other gliders. For the 40×80 grid and the 120 generations shown above, the gliders have not yet looped around, so the behavior matches the infinite-grid case.

**Wolfram's classification** (1984) places GoL in Class IV: complex, localized structures that exhibit indefinite growth, movement, and interaction — distinct from Class I (uniform), Class II (periodic), and Class III (chaotic).